# 02 — Cluster Filtering and Statistics

## Glossary: what each count means

There are four distinct populations; it's easy to confuse them.

| Population | Count | What it is |
|---|---|---|
| Total PDB | ~220K–240K X-ray+other | All structures ever deposited in the PDB (we don't use this directly) |
| npz dataset (manifest) | **238,857** | Structures pre-processed into `.npz` files on this cluster. A filtered subset of the full PDB: passes chain-length, clash, and other Boltz pipeline filters. This is our working dataset. |
| Unique PDB IDs in cluster file | **1,316,075** | Distinct PDB IDs appearing in the RCSB 100%-identity cluster file. Larger than the manifest because the cluster file covers more of the PDB (including structures the Boltz pipeline filtered out). |
| Cluster file tokens | **1,597,265** | `PDBID_entityID` pairs — one per polymer entity per structure per cluster. A single PDB entry can appear multiple times if it has multiple polymer entities that each independently fall into a cluster. |
| Clusters (sequence families) | **1,049,376** | Lines in the cluster file. Each line groups all structures (at the entity level) with ≥ 100% sequence identity to each other. |

**Why tokens > unique PDB IDs:** A structure like a heterodimer has two protein chains (two entities). If both chains happen to cluster together at 100% identity (e.g. both are in the same protein family), both appear in the same cluster line as `1ABC_1` and `1ABC_2`. One PDB, two tokens.

**Why unique PDB IDs in cluster file > manifest:** The RCSB cluster file includes ~1.3M PDB entries; the Boltz npz dataset has 238K. The pipeline applies filters (minimum chain length ≥ 4, consecutive Cα distance, clash detection, etc.) that drop many entries.

**What we filter to:** For this project we need one structure per cluster member = one PDB entry (not one entity). So we deduplicate tokens to unique PDB IDs within each cluster, then apply quality filters.

---

**Filters applied:**
1. PDB is in our npz dataset (manifest)
2. Method == X-ray diffraction
3. Resolution ≤ 2.0 Å (also tested at 2.5 Å)
4. Structure has deposited waters
5. Cluster has ≥ 5 qualifying PDB entries after filtering

In [ ]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, '../scripts')
from cluster_filter import load_manifest_index, filter_clusters, CONSERVATION_DIR, CLUSTER_FILE

print('Loading manifest...')
manifest_index = load_manifest_index()
print(f'Manifest entries (npz dataset): {len(manifest_index):,}')

## 1. Cluster file scale

In [ ]:
# Count clusters, tokens, and unique PDB IDs directly from the file
total_clusters = 0
total_tokens = 0
unique_pdbs_in_file = set()

for line in open(CLUSTER_FILE):
    tokens = line.split()
    if not tokens:
        continue
    total_clusters += 1
    total_tokens += len(tokens)
    for tok in tokens:
        unique_pdbs_in_file.add(tok.rsplit('_', 1)[0].lower())

print('RCSB 100%-identity cluster file:')
print(f'  Clusters (lines / sequence families):  {total_clusters:>12,}')
print(f'  Tokens (PDBID_entityID pairs):          {total_tokens:>12,}')
print(f'  Unique PDB IDs in file:                 {len(unique_pdbs_in_file):>12,}')
print()
print(f'npz manifest entries:                     {len(manifest_index):>12,}')
print(f'PDB IDs in cluster file NOT in manifest:  {len(unique_pdbs_in_file - set(manifest_index)):>12,}')
print(f'PDB IDs in manifest NOT in cluster file:  {len(set(manifest_index) - unique_pdbs_in_file):>12,')
print()
print('(The cluster file covers ~1.3M PDB IDs; the manifest has ~239K. The gap is structures')
print(' the Boltz pipeline filtered out, plus structures with no polymer chain in any cluster.)')

## 2. Quality filtering — full token-level accounting

In [ ]:
print('Filtering at 2.0 Å (min_cluster_size=1 to see full distribution)...')
_, stats_2A = filter_clusters(manifest_index, resolution_cutoff=2.0, min_cluster_size=1)
print('Filtering at 2.5 Å...')
_, stats_25A = filter_clusters(manifest_index, resolution_cutoff=2.5, min_cluster_size=1)
print('Done.')

In [ ]:
def print_accounting(stats, label):
    """Print the two-level accounting separately: token-level and cluster-level."""
    print(f'=== {label} ===')
    print()
    print('--- Token-level accounting (PDBID_entityID pairs) ---')
    print(f'  Total tokens in cluster file:          {stats["total_tokens"]:>10,}')
    print(f'  Same PDB, diff entity (dedup removed): {stats["dups_within_cluster"]:>10,}')
    print(f'  Not in npz manifest:                   {stats["dropped_not_in_manifest"]:>10,}')
    print(f'  Not X-ray diffraction:                 {stats["dropped_not_xray"]:>10,}')
    print(f'  Resolution > {stats["resolution_cutoff"]} Å:                 {stats["dropped_bad_resolution"]:>10,}')
    print(f'  No deposited waters:                   {stats["dropped_no_waters"]:>10,}')
    print(f'  Qualifying PDB-in-cluster entries:     {stats["total_qualifying_pdb_in_clusters"]:>10,}')
    total_check = (stats['dups_within_cluster'] + stats['dropped_not_in_manifest']
                   + stats['dropped_not_xray'] + stats['dropped_bad_resolution']
                   + stats['dropped_no_waters'] + stats['total_qualifying_pdb_in_clusters'])
    print(f'  SUM (must equal total_tokens):         {total_check:>10,}  ✓' if total_check == stats['total_tokens'] else '  SUM MISMATCH')
    print()
    print('--- Cluster-level ---')
    print(f'  Clusters with >=1 qualifying PDB:   {stats["clusters_with_any_qualifying"]:>10,}')
    for size in [5, 10, 20]:
        n = sum(1 for c in stats['qualifying_pdb_counts'] if c >= size)
        print(f'  Clusters with >={size} qualifying PDbs: {n:>10,}')
    print()

print_accounting(stats_2A, '2.0 Å cutoff')
print_accounting(stats_25A, '2.5 Å cutoff')

## 3. Cluster size distributions

In [ ]:
counts_2A = np.array(stats_2A['qualifying_pdb_counts'])
counts_25A = np.array(stats_25A['qualifying_pdb_counts'])

bins = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000, 10000]
bin_labels = ['1', '2-4', '5-9', '10-19', '20-49', '50-99', '100-199', '200-499', '500-999', '>=1000']

h2A, _ = np.histogram(counts_2A, bins=bins)
h25A, _ = np.histogram(counts_25A, bins=bins)

print('Cluster size (qualifying PDbs)  |  2.0 Å  |  2.5 Å')
print('-' * 50)
for label, n2, n25 in zip(bin_labels, h2A, h25A):
    print(f'{label:30s} | {n2:7,} | {n25:7,}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, counts, label, color in [
    (axes[0], counts_2A, '2.0 Å cutoff', 'steelblue'),
    (axes[1], counts_25A, '2.5 Å cutoff', 'darkorange'),
]:
    log_bins = np.logspace(0, 4, 30)
    ax.hist(counts, bins=log_bins, color=color, edgecolor='white', alpha=0.85)
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.axvline(5, color='red', linestyle='--', linewidth=1.2, label='min=5')
    ax.set_xlabel('Cluster size (# qualifying PDB entries)', fontsize=12)
    ax.set_ylabel('# clusters', fontsize=12)
    ax.set_title(f'Cluster size distribution — {label}', fontsize=12)
    ax.legend()
    n_ge5  = (counts >= 5).sum()
    n_ge10 = (counts >= 10).sum()
    n_ge20 = (counts >= 20).sum()
    ax.text(0.62, 0.95, f'>=5:  {n_ge5:,}\n>=10: {n_ge10:,}\n>=20: {n_ge20:,}',
            transform=ax.transAxes, va='top', fontsize=10,
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig(CONSERVATION_DIR / 'cluster_size_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Note on same-PDB / different-entity tokens

The token-level accounting above includes a `dups_within_cluster` bucket. These are tokens where
a PDB entry appears twice in the same cluster line with different entity IDs.

We verified (2026-05-04) that across 3,581 PDBs affected:
- **~1,099** are homodimer-like: both chains have identical sequences (same Boltz cluster ID, same length). RCSB assigned them separate entity IDs, possibly due to minor PTM or terminus differences not reflected in the Boltz preprocessing.
- **~2,124** are heterodimer-like: two genuinely different protein chains that independently fell into the same 100% identity cluster. This can happen for proteins from the same family that appear together in a complex.
- **~358** not in the npz dataset (could not verify).

In both cases, the right behavior is **one entry per PDB structure** — waters belong to the whole structure, not individual chains. Deduplication is correct regardless of the reason.

## 5. Note on missing schema fields

The following fields are **absent** from the preprocessed npz files (see notebook 01):

| Field | Impact |
|---|---|
| Occupancy | Cannot filter alt-conformers by occupancy; they were collapsed at preprocessing |
| Alt-loc identifiers | Same |
| Unit cell / space group | Crystal-contact filtering (step 7) requires raw PDB/mmCIF files, not npz |